# 5. Inspect WorldPop under-18 raster

This notebook inspects the WorldPop 2019 under-age-18 raster before aggregation. The key distinction is between the raster's structure/metadata and the values inside the raster cells.


## Load raster path and inspect spatial metadata

Metadata answers: Where is the raster? What CRS is it in? What are its dimensions, bounds, resolution, NoData value, and dtype?


In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

import numpy as np
import rasterio

from childreach.paths import WORLDPOP_UNDER18_TOTAL_2019_TIF

raster_path = WORLDPOP_UNDER18_TOTAL_2019_TIF

with rasterio.open(raster_path) as src:
    print("Path:", raster_path)
    print("Driver:", src.driver)
    print("CRS:", src.crs)
    print("Width, height:", src.width, src.height)
    print("Band count:", src.count)
    print("Bounds:", src.bounds)
    print("Transform:", src.transform)
    print("Resolution:", src.res)
    print("NoData:", src.nodata)
    print("Data type:", src.dtypes)


## Inspect raster values with NoData masked

`src.nodata` only tells us the missing-value marker. Reading with `masked=True` applies that marker so missing cells do not contaminate summary statistics.


In [ ]:
with rasterio.open(raster_path) as src:
    band = src.read(1, masked=True)

print("Min:", band.min())
print("Max:", band.max())
print("Mean:", band.mean())
print("Masked / NoData cells:", np.ma.count_masked(band))
print("Valid cells:", band.count())


## Interpretation

The raster is ready for zonal statistics only after confirming CRS, bounds, NoData handling, and plausible values. For the final paper, we still need to cite/confirm that the `CN` raster represents estimated population counts per cell, because that assumption justifies summing cell values into ADM2 totals.
